# Створюємо PySpark клієнт

In [28]:
import sys
import os

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

# Створюємо сесію Spark
spark = (
    SparkSession.builder
    .appName("MyGoitSparkSandbox")
    .config("spark.python.use.daemon", "false")
    .config("spark.python.worker.reuse", "false")
    .config("spark.sql.shuffle.partitions", "4")
    # Disable Arrow to avoid Python 3.13 serialization crashes
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.sql.execution.arrow.pyspark.fallback.enabled", "true")
    # Increase timeouts so workers aren't killed during shuffle
    .config("spark.network.timeout", "800s")
    .config("spark.executor.heartbeatInterval", "200s")
    .config("spark.python.worker.timeout", "120")
    .getOrCreate()
)

# Завантажуємо дані

In [29]:
from pathlib import Path

data_dir = Path().resolve() / "data"
users_csv = data_dir / "users.csv"
purchases_csv = data_dir / "purchases.csv"
products_csv = data_dir / "products.csv"

users_df = spark.read.csv(str(users_csv), header=True, inferSchema=True)
purchases_df = spark.read.csv(str(purchases_csv), header=True, inferSchema=True)
products_df = spark.read.csv(str(products_csv), header=True, inferSchema=True)

users_df.createOrReplaceTempView("users")
purchases_df.createOrReplaceTempView("purchases")
products_df.createOrReplaceTempView("products")

# Завдання
## Очистіть дані, видаляючи будь-які рядки з пропущеними значеннями

In [30]:
from pyspark.sql.functions import count, when, col

users_df.select([count(when(col(c).isNull(), c)).alias(c) for c in users_df.columns]).show()
purchases_df.select([count(when(col(c).isNull(), c)).alias(c) for c in purchases_df.columns]).show()
products_df.select([count(when(col(c).isNull(), c)).alias(c) for c in products_df.columns]).show()

+-------+----+---+-----+
|user_id|name|age|email|
+-------+----+---+-----+
|      0|   2|  2|    1|
+-------+----+---+-----+

+-----------+-------+----------+----+--------+
|purchase_id|user_id|product_id|date|quantity|
+-----------+-------+----------+----+--------+
|          0|      2|         1|   1|       1|
+-----------+-------+----------+----+--------+

+----------+------------+--------+-----+
|product_id|product_name|category|price|
+----------+------------+--------+-----+
|         1|           1|       1|    1|
+----------+------------+--------+-----+



In [31]:
users_df = users_df.dropna()
purchases_df = purchases_df.dropna()
products_df = products_df.dropna()

In [32]:
users_df.select([count(when(col(c).isNull(), c)).alias(c) for c in users_df.columns]).show()
purchases_df.select([count(when(col(c).isNull(), c)).alias(c) for c in purchases_df.columns]).show()
products_df.select([count(when(col(c).isNull(), c)).alias(c) for c in products_df.columns]).show()

+-------+----+---+-----+
|user_id|name|age|email|
+-------+----+---+-----+
|      0|   0|  0|    0|
+-------+----+---+-----+

+-----------+-------+----------+----+--------+
|purchase_id|user_id|product_id|date|quantity|
+-----------+-------+----------+----+--------+
|          0|      0|         0|   0|       0|
+-----------+-------+----------+----+--------+

+----------+------------+--------+-----+
|product_id|product_name|category|price|
+----------+------------+--------+-----+
|         0|           0|       0|    0|
+----------+------------+--------+-----+



## Визначте загальну суму покупок за кожною категорією продуктів

In [33]:
from pyspark.sql.functions import sum as spark_sum
(purchases_df.join(products_df, products_df.product_id == purchases_df.product_id)
 .groupBy("category").agg(spark_sum("quantity").alias("total_quantity")).show())

+-----------+--------------+
|   category|total_quantity|
+-----------+--------------+
|     Beauty|            58|
|       Home|           197|
|Electronics|           218|
|   Clothing|           152|
|     Sports|           296|
+-----------+--------------+



## Визначте суму покупок за кожною категорією продуктів для вікової категорії від 18 до 25 включно

In [36]:
(purchases_df.join(products_df, products_df.product_id == purchases_df.product_id)
 .join(users_df, users_df.user_id == purchases_df.user_id)
 .where((col('age') >= 18) & (col('age') <= 25))
 .groupBy("category")
 .agg(spark_sum("quantity").alias("total_quantity"))
 .show())

+-----------+--------------+
|   category|total_quantity|
+-----------+--------------+
|     Beauty|             5|
|       Home|            42|
|Electronics|            31|
|   Clothing|            45|
|     Sports|            53|
+-----------+--------------+



## Визначте частку покупок за кожною категорією товарів від сумарних витрат для вікової категорії від 18 до 25 років.

In [38]:
total_spent = (purchases_df.join(products_df, products_df.product_id == purchases_df.product_id)
 .join(users_df, users_df.user_id == purchases_df.user_id)
 .where((col('age') >= 18) & (col('age') <= 25))
 .withColumn("total_spend", col("quantity") * col("price")).agg(spark_sum("total_spend").alias("total_spend"))).first()[0]

print(f"Total spent: {total_spent}")

Total spent: 1207.6


In [40]:
(purchases_df.join(products_df, products_df.product_id == purchases_df.product_id)
 .join(users_df, users_df.user_id == purchases_df.user_id)
 .where((col('age') >= 18) & (col('age') <= 25))
 .withColumn("total_particles", col("quantity") * col("price") / total_spent)
 .groupBy("category")
 .agg(spark_sum("total_particles").alias("total_particles"))
 .show())

+-----------+--------------------+
|   category|     total_particles|
+-----------+--------------------+
|     Beauty|0.034282875124213325|
|       Home|  0.2990228552500828|
|Electronics| 0.20669095727061942|
|   Clothing|  0.2028817489234846|
|     Sports| 0.25712156343159986|
+-----------+--------------------+



## Виберіть 3 категорії продуктів з найвищим відсотком витрат споживачами віком від 18 до 25 років

In [45]:
from pyspark.sql.functions import round as spark_round

(purchases_df.join(products_df, products_df.product_id == purchases_df.product_id)
 .join(users_df, users_df.user_id == purchases_df.user_id)
 .where((col('age') >= 18) & (col('age') <= 25))
 .withColumn("total_particles", col("quantity") * col("price") / total_spent)
 .groupBy("category")
 .agg(spark_sum("total_particles").alias("total_particles"))
 .withColumn("percentage", spark_round(col("total_particles") * 100, 2))
 .orderBy(col("percentage").desc()).select("category", "percentage")
 .limit(3)
 .show())

+-----------+----------+
|   category|percentage|
+-----------+----------+
|       Home|      29.9|
|     Sports|     25.71|
|Electronics|     20.67|
+-----------+----------+



In [46]:
spark.stop()